# VN Stock Market - ETL Pipeline

Notebook này làm sạch dữ liệu crawl từ CafeF và chuẩn bị file để nạp vào SQL Server.

- **Input:** [`LichSuGia_Crawled.csv`](./LichSuGia_Crawled.csv), dữ liệu nguồn gồm 7.932 dòng.
- **Output:** [`StockPrice_Clean.csv`](./StockPrice_Clean.csv), 7.920 dòng sạch: 396 mã × 20 phiên, từ 09/09/2026 đến 06/10/2026.
- **SQL:** [`01_star_schema.sql`](./01_star_schema.sql) tạo mô hình dữ liệu; [`02_business_questions.sql`](./02_business_questions.sql) chứa truy vấn phân tích.

Chạy notebook với thư mục làm việc là thư mục workspace `Stock_price` để các đường dẫn tương đối tới CSV hoạt động.

In [5]:
import pandas as pd

CSV_PATH = "LichSuGia_Crawled.csv"
OUTPUT_CSV = "StockPrice_Clean.csv"

CONN_STRING = None

# Mapping tên cột gốc (tiếng Việt, trong CSV) -> tên cột sạch (tiếng Anh)
COLUMN_MAP = {
    "Mã": "Ticker",
    "Ngày": "TradeDate",
    "Giá đóng cửa": "ClosePrice",
    "Giá điều chỉnh": "AdjClosePrice",
    "Thay đổi": "ChangeText",
    "Khối lượng khớp lệnh": "MatchedVolume",
    "Giá trị khớp lệnh": "MatchedValue",
    "Khối lượng thỏa thuận": "DealVolume",
    "Giá trị thỏa thuận": "DealValue",
    "Giá mở cửa": "OpenPrice",
    "Giá cao nhất": "HighPrice",
    "Giá thấp nhất": "LowPrice",
}

FINAL_COLUMNS = [
    "Ticker", "TradeDate", "DayOfWeek",
    "OpenPrice", "HighPrice", "LowPrice", "ClosePrice", "AdjClosePrice",
    "MatchedVolume", "MatchedValue", "DealVolume", "DealValue",
]

## 2. EXTRACT

Đọc [`LichSuGia_Crawled.csv`](./LichSuGia_Crawled.csv) bằng UTF-8-SIG. Snapshot hiện có 7.932 dòng thô, gồm 398 mã; dữ liệu thô có cột thỏa thuận trùng lặp và các cột `Unnamed` do cấu trúc file crawl.

In [6]:
def extract(csv_path: str) -> pd.DataFrame: 
    print(f" Đang đọc file: {csv_path}")
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    print(f" Đọc được {len(df):,} dòng, {df.shape[1]} cột.")
    return df

df_raw = extract(CSV_PATH)
df_raw.head()


 Đang đọc file: LichSuGia_Crawled.csv
 Đọc được 7,932 dòng, 20 cột.


,Mã,Ngày,Giá đóng cửa,Giá điều chỉnh,Thay đổi,Khối lượng khớp lệnh,Giá trị khớp lệnh,Khối lượng thỏa thuận,Giá trị thỏa thuận,Khối lượng thỏa thuận.1,Giá trị thỏa thuận.1,Giá mở cửa,Giá cao nhất,Giá thấp nhất,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,AAA,06/10/2026,7.03,7.03,"-0,01 (-0,14%)",2438000,17.04,0,0.0,0,0.0,7.06,7.13,6.92,NaN,NaN,NaN,NaN,NaN,NaN
1,AAA,05/10/2026,7.04,7.04,"-0,01 (-0,14%)",912500,6.46,0,0.0,0,0.0,7.06,7.18,7.04,NaN,NaN,NaN,NaN,NaN,NaN
2,AAA,02/10/2026,7.05,7.05,"-0,11 (-1,54%)",1540600,10.94,0,0.0,0,0.0,7.16,7.17,7.05,NaN,NaN,NaN,NaN,NaN,NaN
3,AAA,01/10/2026,7.16,7.16,"0 (0,00%)",1089400,7.81,0,0.0,0,0.0,7.20,7.22,7.13,NaN,NaN,NaN,NaN,NaN,NaN
4,AAA,30/09/2026,7.16,7.16,"-0,10 (-1,38%)",1021900,7.37,0,0.0,0,0.0,7.27,7.29,7.16,NaN,NaN,NaN,NaN,NaN,NaN


## 3. TRANSFORM

- Đổi tên các cột nguồn tiếng Việt sang schema tiếng Anh; chỉ giữ các cột trong `FINAL_COLUMNS`, bỏ cột trùng/không dùng.
- Parse ngày theo `%d/%m/%Y`, ép kiểu số, chuẩn hóa mã và tạo `DayOfWeek`.
- Loại bản ghi trùng `(Ticker, TradeDate)` và bản ghi thiếu khóa/giá đóng cửa.
- Giữ các mã có số phiên bằng mức tối đa trong dữ liệu. Với snapshot hiện tại, bước này loại BCG và TCD (mỗi mã chỉ có 6 phiên), còn **396 mã × 20 phiên = 7.920 dòng**.

Các chỉ số lợi suất ở bước phân tích nên dùng `AdjClosePrice`, không dùng `ClosePrice` chưa điều chỉnh.

In [7]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    missing = set(COLUMN_MAP) - set(df.columns)
    if missing:
        raise ValueError(f"CSV thiếu các cột kỳ vọng: {missing}")

    print("[TRANSFORM] Đổi tên cột tiếng Việt -> tiếng Anh...")
    df = df.rename(columns=COLUMN_MAP)

    print("Parse cột ngày (dd/mm/yyyy) -> datetime...")
    df["TradeDate"] = pd.to_datetime(df["TradeDate"], format="%d/%m/%Y")

    print("Ép kiểu số cho các cột giá/khối lượng/giá trị...")
    numeric_cols = ["ClosePrice", "AdjClosePrice", "MatchedVolume", "MatchedValue",
                     "DealVolume", "DealValue", "OpenPrice", "HighPrice", "LowPrice"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print("Chuẩn hoá cột Ticker (trim khoảng trắng, upper-case)...")
    df["Ticker"] = df["Ticker"].astype(str).str.strip().str.upper()

    print("Tính thêm cột phái sinh: DayOfWeek...")
    df["DayOfWeek"] = df["TradeDate"].dt.day_name()

    before = len(df)
    df = df.drop_duplicates(subset=["Ticker", "TradeDate"])
    df = df.dropna(subset=["Ticker", "TradeDate", "ClosePrice"])
    session_counts = df.groupby("Ticker")["TradeDate"].transform("nunique")
    df = df[session_counts == session_counts.max()]
    after = len(df)
    if before != after:
        print(f"Loại {before - after} dòng trùng lặp / thiếu dữ liệu / mã không đủ phiên.")

    df = df.sort_values(["Ticker", "TradeDate"]).reset_index(drop=True)
    df = df[FINAL_COLUMNS]

    print(f"[TRANSFORM] Hoàn tất. {len(df):,} dòng sạch, {df['Ticker'].nunique()} mã.")
    return df

df_clean = transform(df_raw)
df_clean.head()

[TRANSFORM] Đổi tên cột tiếng Việt -> tiếng Anh...
Parse cột ngày (dd/mm/yyyy) -> datetime...
Ép kiểu số cho các cột giá/khối lượng/giá trị...
Chuẩn hoá cột Ticker (trim khoảng trắng, upper-case)...
Tính thêm cột phái sinh: DayOfWeek...
Loại 12 dòng trùng lặp / thiếu dữ liệu / mã không đủ phiên.
[TRANSFORM] Hoàn tất. 7,920 dòng sạch, 396 mã.


,Ticker,TradeDate,DayOfWeek,OpenPrice,HighPrice,LowPrice,ClosePrice,AdjClosePrice,MatchedVolume,MatchedValue,DealVolume,DealValue
0,AAA,2026-09-09,Wednesday,7.16,7.45,7.14,7.35,7.35,4275800,31.29,0,0.00
1,AAA,2026-09-10,Thursday,7.35,7.35,7.21,7.30,7.30,1136900,8.28,0,0.00
2,AAA,2026-09-11,Friday,7.22,7.44,7.16,7.17,7.17,2197500,15.97,0,0.00
3,AAA,2026-09-14,Monday,7.15,7.49,7.15,7.39,7.39,3703200,27.24,350000,2.33
4,AAA,2026-09-15,Tuesday,7.50,7.60,7.38,7.44,7.44,2316300,17.32,0,0.00


## 4. Kiểm tra dữ liệu sau Transform

Với snapshot hiện tại, kết quả kỳ vọng là **7.920 dòng, 396 mã, mỗi mã 20 phiên**, từ 09/09/2026 đến 06/10/2026. Kiểm tra kiểu dữ liệu, số phiên tối thiểu/tối đa, khoảng ngày và null ở các cột khóa trước khi load.

In [8]:
print("Data Check:")
print(df_clean.dtypes)
print()
print("Số dòng theo mã (min/max):", df_clean.groupby('Ticker').size().min(), "/",
      df_clean.groupby('Ticker').size().max())
print("Khoảng ngày:", df_clean['TradeDate'].min().date(), "->", df_clean['TradeDate'].max().date())
print("Có null ở cột bắt buộc không:", df_clean[['Ticker','TradeDate','ClosePrice']].isnull().any().any())


Data Check:
Ticker                      str
TradeDate        datetime64[us]
DayOfWeek                   str
OpenPrice               float64
HighPrice               float64
LowPrice                float64
ClosePrice              float64
AdjClosePrice           float64
MatchedVolume             int64
MatchedValue            float64
DealVolume                int64
DealValue               float64
dtype: object

Số dòng theo mã (min/max): 20 / 20
Khoảng ngày: 2026-09-09 -> 2026-10-06
Có null ở cột bắt buộc không: False


## 5. LOAD

Ghi `StockPrice_Clean.csv` vào thư mục làm việc (UTF-8-SIG). Nếu cấu hình `CONN_STRING`, notebook cũng thay thế bảng `dbo.StockPrice_Clean` trên SQL Server; mặc định `CONN_STRING = None` nên chỉ xuất CSV.

In [9]:
def load(df: pd.DataFrame, output_csv: str, conn_string: str = None) -> None:
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"[LOAD] Đã lưu dữ liệu sạch ra file: {output_csv}")

    if conn_string:
        from sqlalchemy import create_engine
        engine = create_engine(conn_string)
        df.to_sql("StockPrice_Clean", engine, schema="dbo",
                   if_exists="replace", index=False, chunksize=1000)
        print("[LOAD] Đã nạp dữ liệu vào bảng dbo.StockPrice_Clean trên SQL Server.")
    else:
        print("[LOAD] Chưa cấu hình CONN_STRING -> bỏ qua bước nạp vào SQL Server.")

load(df_clean, OUTPUT_CSV, CONN_STRING)


[LOAD] Đã lưu dữ liệu sạch ra file: StockPrice_Clean.csv
[LOAD] Chưa cấu hình CONN_STRING -> bỏ qua bước nạp vào SQL Server.
